In [ ]:
# ========================================================
# BÀI 7 (TRANG 27 - 31)
# Đề bài: Xây dựng ứng dụng nhận dạng số viết tay dựa trên dữ liệu MNIST.
# Từng bước thực hiện:
# - Chuẩn bị thư viện cần thiết
# - Tải dữ liệu load_digits từ sklearn
# - Biểu diễn 4 mẫu dữ liệu huấn luyện tương ứng với 4 lớp 0, 1, 2, 3
# - Flatten ảnh 8x8 thành vector 1x64
# - Chia tập dữ liệu 80% train, 20% test
# - Xây dựng cây quyết định ID3 (criterion='entropy', random_state=33)
# - Dự đoán nhãn kiểm thử và biểu diễn một vài kết quả
# - Đánh giá: Accuracy, Classification report, Confusion matrix Heatmap
# - Ứng dụng đọc ảnh chữ số viết tay bên ngoài, làm rõ nét ảnh bằng PIL
#   và nhận diện số bằng cây ID3.
# ========================================================

# 1. BẮT BUỘC: IN HỌ TÊN VÀ MSSV
HO_TEN = "Đào Tiến Đạt"
MSSV = "6451071017"
print(f"HỌ TÊN: {HO_TEN} - MSSV: {MSSV}")
print("=" * 60)

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from sklearn import datasets, tree, metrics
from sklearn.model_selection import train_test_split
from PIL import Image, ImageOps
import os
import pandas as pd
import numpy as np

def find_file(name):
    paths = [
        name,
        os.path.join(r"D:\khai pha du lieu\Lab2\Dataset-20261103T094921Z-1-001\Dataset", name),
        os.path.join("..", "..", "..", "Dataset-20261103T094921Z-1-001", "Dataset", name),
        os.path.join("Dataset", name)
    ]
    for p in paths:
        if os.path.exists(p):
            return p
    return name


# --------------------------------------------------------
# Bước 1 & 2: Tải dữ liệu chữ số MNIST có sẵn trong sklearn
# --------------------------------------------------------
digits = datasets.load_digits()
print(f"Tổng số mẫu ảnh chữ số: {len(digits.images)}")
print(f"Kích thước mỗi ảnh gốc: {digits.images[0].shape}")

# --------------------------------------------------------
# Bước 3: Biểu diễn 4 mẫu dữ liệu tương ứng 4 lớp 0, 1, 2, 3
# --------------------------------------------------------
images_and_labels = list(zip(digits.images, digits.target))
plt.figure(figsize=(8, 3))
for index, (image, label) in enumerate(images_and_labels[:4]):
    plt.subplot(1, 4, index + 1)
    plt.axis('off')
    plt.imshow(image, cmap=plt.cm.gray_r, interpolation='nearest')
    plt.title(f'Training: {label}')
plt.tight_layout()
plt.show()

# --------------------------------------------------------
# Bước 4: Chuyển đổi mỗi đối tượng 8x8 thành vector 1x64
# --------------------------------------------------------
n_samples = len(digits.images)
data = digits.images.reshape((n_samples, -1))
print(f"Kích thước ma trận đặc trưng sau khi reshape: {data.shape}")

# --------------------------------------------------------
# Bước 5: Chia dữ liệu: 80% huấn luyện, 20% kiểm thử (shuffle=False)
# --------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    data, digits.target, test_size=0.2, shuffle=False
)
print(f"Kích thước tập huấn luyện: {X_train.shape}")
print(f"Kích thước tập kiểm thử: {X_test.shape}")

# --------------------------------------------------------
# Bước 6: Xây dựng và huấn luyện cây quyết định ID3
# --------------------------------------------------------
classifier = tree.DecisionTreeClassifier(criterion="entropy", random_state=33)
classifier.fit(X_train, y_train)

# --------------------------------------------------------
# Bước 7: Dự đoán và biểu diễn một vài kết quả kiểm thử
# --------------------------------------------------------
y_pred = classifier.predict(X_test)

# Biểu diễn trực quan 4 mẫu kiểm thử kèm nhãn dự đoán
images_and_predictions = list(zip(digits.images[len(X_train):], y_pred))
plt.figure(figsize=(8, 3))
for index, (image, prediction) in enumerate(images_and_predictions[:4]):
    plt.subplot(1, 4, index + 1)
    plt.axis('off')
    plt.imshow(image, cmap=plt.cm.gray_r, interpolation='nearest')
    plt.title(f'Pred: {prediction}')
plt.tight_layout()
plt.show()

# --------------------------------------------------------
# Bước 8: Đánh giá thông số đạt được và ma trận nhầm lẫn
# --------------------------------------------------------
acc_score = metrics.accuracy_score(y_test, y_pred)
print(f"Độ chính xác (Accuracy score): {acc_score:.4f}")
print("-" * 60)
print("Classification report:")
print(metrics.classification_report(y_test, y_pred))

# Ma trận nhầm lẫn Heatmap
disp = metrics.ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues')
disp.figure_.suptitle("Ma trận nhầm lẫn (Confusion Matrix) - ID3 Digit Recognizer")
plt.show()

# --------------------------------------------------------
# Bước 9: Nhận diện ảnh chữ số viết tay chụp từ bên ngoài
# --------------------------------------------------------
print("=" * 60)
print("ỨNG DỤNG NHẬN DẠNG CHỮ SỐ VIẾT TAY TỪ ẢNH CHỤP BÊN NGOÀI:")

test_img_path = find_file("test_image_Bai7.jpg")
if not os.path.exists(test_img_path):
    test_img_path = find_file("test_image_Bai7(1).jpg")

print(f"Đường dẫn ảnh kiểm thử: {test_img_path}")

# 1. Đọc ảnh, chuyển về ảnh xám (L) và resize về kích thước 8x8
img = Image.open(test_img_path).convert("L").resize((8, 8))

# 2. Đảo màu (để nét chữ sáng màu hơn nền tương thích với digits dataset)
img_inverted = ImageOps.invert(img)
im2arr = np.array(img_inverted)

# Hiển thị ảnh sau khi resize và invert
plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plt.imshow(im2arr, cmap=plt.cm.gray_r, interpolation='nearest')
plt.title("Ảnh sau resize (8x8)")

# 3. Làm rõ nét ảnh bằng cách phân ngưỡng nhị phân (ngưỡng 110) và scale về thang [0, 16]
threshold = 110
im2arr_norm = np.where(im2arr > threshold, 16.0, 0.0)

plt.subplot(1, 2, 2)
plt.imshow(im2arr_norm, cmap=plt.cm.gray_r, interpolation='nearest')
plt.title("Ảnh sau làm rõ (Threshold=110)")
plt.show()

# 4. Tiến hành nhận diện bằng cây ID3 và in kết quả
img1d = im2arr_norm.reshape(1, -1)
predicted_digit = classifier.predict(img1d)

print(f"==> KẾT QUẢ DỰ ĐOÁN CHỮ SỐ VIẾT TAY: {predicted_digit[0]}")
